# Plant Disease Detection — retrain ResNet18 (Google Colab, free GPU)

This notebook rebuilds the model for the live demo:

1. Downloads the public **PlantVillage** dataset (23 classes: Apple, Corn, Pepper, Potato, Tomato)
2. Fine-tunes a pretrained **ResNet18** with PyTorch (transfer learning)
3. Exports the model to **ONNX** (small and fast on a free CPU server)
4. Uploads the model to the **Hugging Face Space** demo (or downloads a zip)

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Takes about 25–35 minutes.

In [ ]:
# 1) Setup
!pip -q install onnx onnxscript huggingface_hub
import os, json, time, random, shutil, re, subprocess
import torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| torch", torch.__version__)
assert DEVICE.type == "cuda", "Switch to a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# 2) Download only the 23 needed classes from PlantVillage (github.com/spMohanty/PlantVillage-Dataset)
CLASS_NAMES = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight", "Corn_(maize)___healthy",
    "Pepper__bell___Bacterial_spot", "Pepper__bell___healthy",
    "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
    "Tomato_Bacterial_spot", "Tomato_Early_blight", "Tomato_Late_blight", "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot", "Tomato_Spider_mites_Two_spotted_spider_mite", "Tomato__Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus", "Tomato__Tomato_mosaic_virus", "Tomato_healthy",
]
norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())

REPO = "/content/PlantVillage-Dataset"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/spMohanty/PlantVillage-Dataset.git", REPO], check=True)
folders = subprocess.run(["git", "-C", REPO, "ls-tree", "-d", "--name-only", "HEAD", "raw/color/"],
                         capture_output=True, text=True, check=True).stdout.split("\n")
folders = [f.split("/")[-1] for f in folders if f.strip()]
lookup = {norm(f): f for f in folders}
mapping = {c: lookup.get(norm(c)) for c in CLASS_NAMES}
missing = [c for c, f in mapping.items() if f is None]
assert not missing, f"Could not match: {missing}\nAvailable: {folders}"

subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", "--no-cone"] +
               [f"raw/color/{f}/" for f in mapping.values()], check=True)

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)
for cls, folder in mapping.items():
    dst = os.path.join(DATA_DIR, cls)
    if not os.path.exists(dst):
        os.symlink(os.path.join(REPO, "raw/color", folder), dst)
counts = {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in CLASS_NAMES}
print("Images:", sum(counts.values()))
for c, n in counts.items(): print(f"  {n:5d}  {c}")

In [ ]:
# 3) Data loaders (80% train / 20% validation, fixed seed)
IMG = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

base = datasets.ImageFolder(DATA_DIR)
assert base.classes == sorted(CLASS_NAMES)
classes = base.classes
idx = list(range(len(base)))
random.Random(42).shuffle(idx)
n_val = int(0.2 * len(idx))
val_idx, train_idx = idx[:n_val], idx[n_val:]
train_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx)
val_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf), val_idx)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print("Train:", len(train_ds), "Val:", len(val_ds), "Classes:", len(classes))

In [ ]:
# 4) Train ResNet18: 2 epochs head only, then fine-tune the whole network
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.amp.GradScaler("cuda")

def run_epoch(train):
    model.train(train)
    dl = train_dl if train else val_dl
    loss_sum, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(x); loss = criterion(out, y)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item(); total += y.size(0)
    return loss_sum / total, correct / total

for p in model.parameters(): p.requires_grad = False
for p in model.fc.parameters(): p.requires_grad = True
opt = optim.AdamW(model.fc.parameters(), lr=1e-3)

HEAD_EPOCHS, FT_EPOCHS = 2, 5
history, best_acc, t0 = [], 0.0, time.time()
for epoch in range(1, HEAD_EPOCHS + FT_EPOCHS + 1):
    if epoch == HEAD_EPOCHS + 1:
        for p in model.parameters(): p.requires_grad = True
        opt = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
        print("Unfroze the full network for fine-tuning")
    tl, ta = run_epoch(True)
    vl, va = run_epoch(False)
    if epoch > HEAD_EPOCHS: sched.step()
    history.append(dict(epoch=epoch, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va))
    flag = ""
    if va > best_acc:
        best_acc = va; torch.save(model.state_dict(), "/content/best_resnet18.pth"); flag = "  <- best, saved"
    print(f"Epoch {epoch}: train_acc={ta:.4f} val_acc={va:.4f} val_loss={vl:.4f}{flag}")
print(f"Best validation accuracy: {best_acc*100:.2f}%  ({(time.time()-t0)/60:.1f} min)")

In [ ]:
# 5) Evaluate best model (per-class accuracy)
model.load_state_dict(torch.load("/content/best_resnet18.pth")); model.eval()
correct = torch.zeros(len(classes)); total = torch.zeros(len(classes))
with torch.no_grad():
    for x, y in val_dl:
        pred = model(x.to(DEVICE)).argmax(1).cpu()
        for t, p in zip(y, pred):
            total[t] += 1; correct[t] += int(t == p)
per_class = {c: round(float(correct[i] / total[i]), 4) for i, c in enumerate(classes)}
overall = float(correct.sum() / total.sum())
print(f"Validation accuracy: {overall*100:.2f}% on {int(total.sum())} images")
for c, a in sorted(per_class.items(), key=lambda kv: kv[1]): print(f"  {a*100:6.2f}%  {c}")

In [ ]:
# 6) Export to ONNX (softmax included) + metadata
OUT = "/content/space_model"
os.makedirs(OUT, exist_ok=True)

class WithSoftmax(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return torch.softmax(self.m(x), dim=1)

export_model = WithSoftmax(model.float().cpu()).eval()
torch.onnx.export(export_model, torch.randn(1, 3, IMG, IMG), f"{OUT}/plant_resnet18.onnx",
                  input_names=["input"], output_names=["probs"],
                  dynamic_axes={"input": {0: "batch"}, "probs": {0: "batch"}}, opset_version=17, dynamo=False)
json.dump(classes, open(f"{OUT}/class_names.json", "w"), indent=1)
json.dump({"model": "ResNet18 (ImageNet pretrained, fine-tuned)", "dataset": "PlantVillage (23 classes)",
           "train_images": len(train_ds), "val_images": len(val_ds),
           "val_accuracy": round(overall, 4), "per_class_accuracy": per_class, "history": history},
          open(f"{OUT}/metrics.json", "w"), indent=1)

# sanity check with onnxruntime
!pip -q install onnxruntime
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession(f"{OUT}/plant_resnet18.onnx")
x, y = next(iter(val_dl))
probs = sess.run(None, {"input": x[:16].numpy()})[0]
print("ONNX agrees with labels on", int((probs.argmax(1) == y[:16].numpy()).sum()), "/ 16 samples")
print("Files:", os.listdir(OUT))
print(f"Model size: {os.path.getsize(f'{OUT}/plant_resnet18.onnx')/1e6:.1f} MB")

## 7) Put the model into your Hugging Face Space

**Option A — upload straight to the Space (recommended).** The demo is a free *Static* Hugging Face Space; the model runs in the
visitor's browser with ONNX Runtime Web. Make a token at huggingface.co → Settings → Access Tokens → *New token* (type **Write**)
and paste it only into the prompt below — never into code or chat. Press Enter at the Space id prompt to use the default.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi
SPACE_ID = input("Space id [Praneeth9392/plant-disease-detection]: ").strip() or "Praneeth9392/plant-disease-detection"
api = HfApi(token=getpass("Hugging Face write token: "))
api.upload_folder(folder_path=OUT, path_in_repo="model", repo_id=SPACE_ID, repo_type="space",
                  commit_message=f"Add ResNet18 ONNX model (val acc {overall*100:.2f}%)")
print("Uploaded. The Space will rebuild in a minute: https://huggingface.co/spaces/" + SPACE_ID)

**Option B — download a zip** and upload the `model` folder to the Space's *Files* tab yourself.

In [ ]:
shutil.make_archive("/content/plant_model", "zip", OUT)
from google.colab import files
files.download("/content/plant_model.zip")